# QC filtering of subsets

In this notebook, we apply data QC filters to each of the major sets of data that were compiled. These include removing cells flagged as Doublets by scrublet, and removal of cells with abnormally low or high gene counts.

## Load libraries

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

from datetime import date
import hisepy
import os
import pandas as pd
import re
import scanpy as sc
import anndata as ad

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

## Set QC Cutoffs

In [2]:
max_mito = 10
min_genes = 200
max_genes = 5000

## Helper functions

These functions make reading HISE .h5ad files straightforward

In [3]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [4]:
def read_adata_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = sc.read_h5ad(cache_file)
    return res

This function will be used to apply the QC filtering criteria, write the output file, and return counts so we can track what happens at each step.

In [5]:
def apply_qc_filters(
    adata, 
    group_name, 
    out_files, 
    max_mito, 
    min_genes, 
    max_genes):

    counts = {
        'group': group_name
    }
    
    # Filter doublets
    counts['n_start'] = [adata.shape[0]]
    counts['n_doublets'] = [sum(adata.obs['predicted_doublet'] == True)]
    adata = adata[adata.obs['predicted_doublet'] == False]
    counts['n_singlets'] = [adata.shape[0]]
    
    # Compute fraction mitochondrial
    adata.var["mito"] = adata.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mito"], inplace=True)
    
    counts['n_high_mito'] = [sum(adata.obs["pct_counts_mito"] >= max_mito)]
    adata = adata[adata.obs["pct_counts_mito"] < max_mito]
    counts['n_low_mito'] = [adata.shape[0]]

    # Remove low gene counts
    counts['n_low_genes'] = [sum(adata.obs['n_genes'] <= min_genes)]
    adata = adata[adata.obs['n_genes'] > min_genes]
    counts['n_above_min_genes'] = [adata.shape[0]]

    # Remove high gene counts
    counts['n_high_genes'] = [sum(adata.obs['n_genes'] >= max_genes)]
    adata = adata[adata.obs['n_genes'] < max_genes]
    counts['n_below_max_genes'] = [adata.shape[0]]
    
    counts['total_removed'] = [counts['n_start'][0] - adata.shape[0]]
    counts['n_final'] = [adata.shape[0]]
    
    adata.write_h5ad(out_files['h5ad_file'])

    obs = adata.obs
    obs.to_csv(out_files['csv_file'])
    obs.to_parquet(out_files['parquet_file'])

    counts_df = pd.DataFrame(counts)
    
    return counts_df

In [6]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Identify files for use in HISE

In [7]:
## Search ID from previous step
search_id = 'manganese-fermium-selenium'

Retrieve files stored in our HISE project store

In [9]:
ps_df = hisepy.list_files_in_project_store('Dyna_IHandA')
ps_df = ps_df[['id', 'name']]

Filter for files from the previous notebook using our search_id

In [10]:
search_df = ps_df[ps_df['name'].str.contains(search_id)]
search_df = search_df.sort_values('name')

Filter for h5ad files

In [11]:
search_df = search_df[search_df['name'].str.contains('.h5ad')]

In [12]:
search_df

,id,name
62,d028a36a-ed18-497a-82ea-dc424aa0b6a2,manganese-fermium-selenium/up1_pbmc_set1_raw_l...
63,76ebda13-fdfa-4f6b-be7b-82f681832b7b,manganese-fermium-selenium/up1_pbmc_set2_raw_l...


In [13]:
h5ad_uuids = {}
for i in range(search_df.shape[0]):
    fn = search_df['name'].tolist()[i]
    group_name = re.sub('.+_pbmc_', '', fn)
    group_name = re.sub('_raw.+', '', group_name)
    h5ad_uuids[group_name] = search_df['id'].tolist()[i]

In [14]:
h5ad_uuids

{'set1': 'd028a36a-ed18-497a-82ea-dc424aa0b6a2',
 'set2': '76ebda13-fdfa-4f6b-be7b-82f681832b7b'}

## Set up output filenames

In [15]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [16]:
out_files = {}
for group_name in h5ad_uuids.keys():
    h5ad_file = 'output/pbmc_{g}_initial_qc_{d}.h5ad'.format(
        g = group_name,
        d = date.today()
    )
    csv_file = 'output/pbmc_{g}_initial_qc_meta_{d}.csv'.format(
        g = group_name,
        d = date.today()
    )
    parquet_file = 'output/pbmc_{g}_initial_qc_meta_{d}.parquet'.format(
        g = group_name,
        d = date.today()
    )
    out_files[group_name] = {
        'h5ad_file': h5ad_file,
        'csv_file': csv_file,
        'parquet_file': parquet_file
    }

#### Debug set4

In [8]:
adata = read_adata_uuid('88246547-88de-4c9b-8335-ad62e058fa52')
print(adata.shape)

(2157122, 33538)


In [ ]:
adata.obs.drop(columns=columns_to_remove, inplace=True)
    group_counts = apply_qc_filters(
        adata, 
        group_name = group_name, 
        out_files = group_out_files, 
        max_mito = max_mito, 
        min_genes = min_genes, 
        max_genes = max_genes)
print(group_counts)

#### Debug error caused by duplicate columns 

In [18]:
adata = read_adata_uuid('45105e60-a0f1-4210-859d-b1c09442df33')
print(adata.shape)

(2087541, 33538)


In [20]:
[sum(adata.obs['predicted_doublet'] == True)]

[20464]

In [28]:
print(adata.obs['predicted_doublet'].dtype)
print(adata.obs['predicted_doublet'].head())

print(type(adata))
print(type(adata.obs))

bool
barcodes
096de8deb2e811eda352f66963206748    False
096df18ab2e811eda352f66963206748    False
096df2fcb2e811eda352f66963206748    False
096e018eb2e811eda352f66963206748    False
096e0472b2e811eda352f66963206748    False
Name: predicted_doublet, dtype: bool
<class 'anndata._core.anndata.AnnData'>
<class 'pandas.core.frame.DataFrame'>


In [32]:
duplicate_columns = adata.obs.columns[adata.obs.columns.duplicated()]
duplicate_columns

Index(['over_clustering', 'majority_voting', 'over_clustering',
       'majority_voting'],
      dtype='object')

In [33]:
columns_to_remove = ['majority_voting', 'over_clustering']

# Remove the specified columns from adata.obs
adata.obs.drop(columns=columns_to_remove, inplace=True)

In [34]:
duplicate_columns = adata.obs.columns[adata.obs.columns.duplicated()]
duplicate_columns

Index([], dtype='object')

In [35]:
adata = adata[adata.obs['predicted_doublet'] == False]

## Apply to each subset

In [17]:
import random

In [18]:
items = list(h5ad_uuids.items())

# Shuffle the list
random.shuffle(items)

# Convert the shuffled list back to a dictionary
h5ad_uuids = dict(items)

print(h5ad_uuids)

{'set1': 'd028a36a-ed18-497a-82ea-dc424aa0b6a2', 'set2': '76ebda13-fdfa-4f6b-be7b-82f681832b7b'}


In [19]:
columns_to_remove = ['majority_voting', 'over_clustering']

In [23]:
filter_counts = []
for group_name, uuid in h5ad_uuids.items():
    print(group_name)
    group_out_files = out_files[group_name]

    adata = read_adata_uuid(uuid)
    print(adata.shape)

    adata.obs.drop(columns=columns_to_remove, inplace=True)
    group_counts = apply_qc_filters(
        adata, 
        group_name = group_name, 
        out_files = group_out_files, 
        max_mito = max_mito, 
        min_genes = min_genes, 
        max_genes = max_genes
    )
    print(group_counts)
    
    filter_counts.append(group_counts)

set1
(1929009, 33538)


/tmp/ipykernel_2576/1886826524.py:20: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["mito"] = adata.var_names.str.startswith("MT-")


  group  n_start  n_doublets  n_singlets  n_high_mito  n_low_mito  \
0  set1  1929009       10843     1918166        63454     1854712   

   n_low_genes  n_above_min_genes  n_high_genes  n_below_max_genes  \
0          773            1853939          2124            1851815   

   total_removed  n_final  
0          77194  1851815  
set2
downloading fileID: 76ebda13-fdfa-4f6b-be7b-82f681832b7b
Files have been successfully downloaded!
(1981919, 33538)


/tmp/ipykernel_2576/1886826524.py:20: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["mito"] = adata.var_names.str.startswith("MT-")


  group  n_start  n_doublets  n_singlets  n_high_mito  n_low_mito  \
0  set2  1981919       10989     1970930        75209     1895721   

   n_low_genes  n_above_min_genes  n_high_genes  n_below_max_genes  \
0         1353            1894368          2820            1891548   

   total_removed  n_final  
0          90371  1891548  


In [24]:
adata

AnnData object with n_obs × n_vars = 1981919 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score'

## Assemble all counts

In [25]:
all_filter_counts = pd.concat(filter_counts)
all_filter_counts

,group,n_start,n_doublets,n_singlets,n_high_mito,n_low_mito,n_low_genes,n_above_min_genes,n_high_genes,n_below_max_genes,total_removed,n_final
0,set1,1929009,10843,1918166,63454,1854712,773,1853939,2124,1851815,77194,1851815
0,set2,1981919,10989,1970930,75209,1895721,1353,1894368,2820,1891548,90371,1891548


In [26]:
counts_file = 'output/pbmc_qc_filter_counts_{d}.csv'.format(d = date.today())
all_filter_counts.to_csv(counts_file)

## Upload assembled data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [29]:
ss = hisepy.get_study_spaces()
print(ss[0]['name'])
print(ss[0]['id'])
study_space_uuid = ss[0]['id']

title = '04 PBMC QC Filtered .h5ad {d}'.format(d = date.today())

UP1 scRNAseq Study
0b6bf907-6985-40e0-944d-677ac932677f


In [30]:
search_id = element_id()
search_id

'hafnium-copper-praseodymium'

In [31]:
in_files = list(h5ad_uuids.values())
in_files

['d028a36a-ed18-497a-82ea-dc424aa0b6a2',
 '76ebda13-fdfa-4f6b-be7b-82f681832b7b']

In [32]:
out_list = []
for file_dict in out_files.values():
    for fn in file_dict.values():
        out_list.append(fn)

In [33]:
out_list = out_list + [counts_file]

In [34]:
out_list

['output/pbmc_set1_initial_qc_2024-08-19.h5ad',
 'output/pbmc_set1_initial_qc_meta_2024-08-19.csv',
 'output/pbmc_set1_initial_qc_meta_2024-08-19.parquet',
 'output/pbmc_set2_initial_qc_2024-08-19.h5ad',
 'output/pbmc_set2_initial_qc_meta_2024-08-19.csv',
 'output/pbmc_set2_initial_qc_meta_2024-08-19.parquet',
 'output/pbmc_qc_filter_counts_2024-08-19.csv']

In [35]:
hisepy.upload.upload_files(
    files = out_list,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['output/pbmc_set1_initial_qc_2024-08-19.h5ad', 'output/pbmc_set1_initial_qc_meta_2024-08-19.csv', 'output/pbmc_set1_initial_qc_meta_2024-08-19.parquet', 'output/pbmc_set2_initial_qc_2024-08-19.h5ad', 'output/pbmc_set2_initial_qc_meta_2024-08-19.csv', 'output/pbmc_set2_initial_qc_meta_2024-08-19.parquet', 'output/pbmc_qc_filter_counts_2024-08-19.csv']. Do you truly want to proceed?


(y/n) y


{'trace_id': '80b41989-cdb7-4324-b710-fa3ba5b898ce',
 'files': ['output/pbmc_set1_initial_qc_2024-08-19.h5ad',
  'output/pbmc_set1_initial_qc_meta_2024-08-19.csv',
  'output/pbmc_set1_initial_qc_meta_2024-08-19.parquet',
  'output/pbmc_set2_initial_qc_2024-08-19.h5ad',
  'output/pbmc_set2_initial_qc_meta_2024-08-19.csv',
  'output/pbmc_set2_initial_qc_meta_2024-08-19.parquet',
  'output/pbmc_qc_filter_counts_2024-08-19.csv']}

In [37]:
import session_info
session_info.show()

In [38]:
adata.obs.head()

,barcodes,batch_id,cell_name,cell_uuid,chip_id,hto_barcode,hto_category,n_genes,n_mito_umis,n_reads,...,file.id,subset_grp,predicted_doublet,doublet_score,AIFI_L1,AIFI_L1_score,AIFI_L2,AIFI_L2_score,AIFI_L3,AIFI_L3_score
barcodes,,,,,,,,,,,,,,,,,,,,,
096de8deb2e811eda352f66963206748,096de8deb2e811eda352f66963206748,B143,slobbery_cozy_feline,096de8deb2e811eda352f66963206748,B143-P2C2,AAGTATCGTTTCGCA,singlet,1736,245,12628,...,366ef40a-2241-4852-a985-14f9ef616964,set1,False,0.028902,Monocyte,1.000000,CD14 monocyte,0.984092,Core CD14 monocyte,1.0
096df18ab2e811eda352f66963206748,096df18ab2e811eda352f66963206748,B143,funny_socalled_dodobird,096df18ab2e811eda352f66963206748,B143-P2C2,AAGTATCGTTTCGCA,singlet,1392,72,18781,...,366ef40a-2241-4852-a985-14f9ef616964,set1,False,0.104286,T cell,0.999997,Memory CD4 T cell,0.998489,CM CD4 T cell,1.0
096df2fcb2e811eda352f66963206748,096df2fcb2e811eda352f66963206748,B143,wolfish_overjoyed_upupa,096df2fcb2e811eda352f66963206748,B143-P2C2,AAGTATCGTTTCGCA,singlet,1085,106,13942,...,366ef40a-2241-4852-a985-14f9ef616964,set1,False,0.017801,T cell,0.999998,Memory CD4 T cell,0.980953,GZMB- CD27- EM CD4 T cell,1.0
096e018eb2e811eda352f66963206748,096e018eb2e811eda352f66963206748,B143,pelage_cashmere_blackfly,096e018eb2e811eda352f66963206748,B143-P2C2,AAGTATCGTTTCGCA,singlet,690,73,4267,...,366ef40a-2241-4852-a985-14f9ef616964,set1,False,0.050804,T cell,0.999872,Memory CD4 T cell,0.613473,CM CD4 T cell,1.0
096e0472b2e811eda352f66963206748,096e0472b2e811eda352f66963206748,B143,halfjoking_baleful_canvasback,096e0472b2e811eda352f66963206748,B143-P2C2,AAGTATCGTTTCGCA,singlet,1211,108,13476,...,366ef40a-2241-4852-a985-14f9ef616964,set1,False,0.145267,T cell,0.999993,Memory CD4 T cell,0.962942,GZMB- CD27+ EM CD4 T cell,1.0


#### Notes
1. Kernel died a few times while processing ~2 million cells, split into smaller chunks?